<div dir="rtl" align="right">

# تحليلُ فورييهَ السريعُ \(FFT\)

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

يُحوّلُ تحويلُ فورييهَ السريعُ الإشارةَ من المجالِ الزمنيِّ إلى المجالِ التردديّ، فيُظهرُ الطاقةَ التي تَحملها الإشارةُ عندَ كلِّ تردد. نُظلّلُ نطاقاتِ موجاتِ الدماغِ الخمسَ (دلتا، ثيتا، ألفا، بيتا، غاما) لِربطِ الطيفِ بالدلالاتِ الفسيولوجيّة.

## المُخرجاتُ المُتوقّعةُ

- قمّةٌ بارزةٌ في نطاقِ ألفا (8-13 Hz) تَعكسُ نشاطَ القشرةِ البصرية
- طاقةٌ في نطاقَي دلتا وثيتا شائعةٌ في بياناتِ EEG الخام
- تَناقصٌ تدريجيٌّ في الطاقةِ معَ زيادةِ الترددِ (قانون 1/f)

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| القناةُ | P4 | المنطقةُ الجداريةُ |
| معدّلُ الأخذِ | 200 Hz | عيّنةٌ كلَّ 5 ms |
| التردداتُ | 0-80 Hz | نطاقُ التحليلِ |
| النطاقاتُ | 5 | دلتا، ثيتا، ألفا، بيتا، غاما |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly wfdb pywt


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2، قناةَ P4 (المنطقةُ الجداريةُ).

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. تطبيقُ تحويلِ فورييهَ السريعِ

نُطبّقُ `scipy.fft.fft` على الإشارةِ الكاملة، ثمّ نَستخرجُ القيمةَ المطلقةَ لِحسابِ السعةِ التردديّة. نُبقي التردداتِ الموجبةَ فقط لأنَّ النصفَ السالبَ يُكرّرُ معلوماتِ النصفِ الموجبِ.

</div>

In [ ]:
from scipy.fft import fft, fftfreq

spectrum = fft(channel_data)
freqs = fftfreq(len(channel_data), 1 / fs)
magnitude = np.abs(spectrum)
pos_mask = freqs >= 0
freqs = freqs[pos_mask]
magnitude = magnitude[pos_mask]
print(f'Frequency range: {freqs.min():.1f} - {freqs.max():.1f} Hz')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- القممُ البارزةُ تُشيرُ إلى التردداتِ الأكثرِ حضورًا في الإشارة
- التظليلُ المُلوّنُ يُحدّدُ نطاقاتِ موجاتِ الدماغِ الخمسَ
- استخدمْ أداةَ التكبيرِ لِفحصِ نطاقاتٍ تردديّةٍ مُحدّدةٍ


</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = np.arange(n_plot) / fs

BANDS = [
    ('Delta', 0.5, 4, 'green'),
    ('Theta', 4, 8, 'blue'),
    ('Alpha', 8, 13, 'orange'),
    ('Beta', 13, 30, 'red'),
    ('Gamma', 30, 80, 'purple'),
]

fig = make_subplots(rows=2, cols=1, shared_xaxes=False,
                    subplot_titles=('Original signal - Channel P4',
                                    'FFT Spectrum - Channel P4'))
fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot], name='Signal',
                         line=dict(color='gray', width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=freqs, y=magnitude, name='Magnitude',
                         line=dict(color='black', width=0.8)), row=2, col=1)
for name, fmin, fmax, color in BANDS:
    fig.add_vrect(x0=fmin, x1=fmax, fillcolor=color, opacity=0.1,
                  line_width=0, row=2, col=1)
fig.update_xaxes(range=[0, 80], row=2, col=1)
fig.update_layout(height=700, title_text='FFT Analysis - Channel P4',
                  xaxis_title='Time (s)', xaxis2_title='Frequency (Hz)',
                  yaxis_title='Amplitude (uV)', yaxis2_title='Magnitude',
                  showlegend=False)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- تحويلُ فورييهَ يَكشفُ المكوّناتِ التردديّةَ الثابتةَ في الإشارة
- قممُ الطيفِ تُقابلُ التردداتِ الأكثرِ حضورًا
- التحويلُ يُعطي متوسطًا للطيفِ على كاملِ المدّةِ ولا يُحدّدُ متى ظَهرَ كلُّ تردد
- تحويلُ المويجاتِ يُعالجُ هذا القصورَ بِتحليلٍ زمنيٍّ-تردديٍّ


</div>